In [2]:
import glob
import pandas as pd
chunk_files = glob.glob("parquet_chunks/*.parquet")

df = pd.read_parquet("/home/tts26/sonh/Orbit-Wars-RL/gnn_rl/parquet_chunks/chunk_00000.parquet")
df.head()

,step,obs_step,player_id,action_source,action_angle,action_ships,raw_obs
0,2,1,0,4.0,-0.985460,13.0,"{'angular_velocity': 0.028384471849050706, 'co..."
1,2,1,2,6.0,-2.556257,13.0,"{'angular_velocity': 0.028384471849050706, 'co..."
2,2,1,3,7.0,2.156132,13.0,"{'angular_velocity': 0.028384471849050706, 'co..."
3,7,6,1,5.0,-0.136370,28.0,"{'angular_velocity': 0.028384471849050706, 'co..."
4,9,8,2,6.0,2.919956,21.0,"{'angular_velocity': 0.028384471849050706, 'co..."


In [9]:
print(df.dtypes)
print(f"Data Shape: {df.shape}")

step               int64
obs_step           int64
player_id          int64
action_source    float64
action_angle     float64
action_ships     float64
raw_obs           object
dtype: object
Data Shape: (371283, 7)


In [ ]:
sample = df["raw_obs"].iloc[0]

print(type(sample))
print(sample)

<class 'dict'>
{'angular_velocity': 0.028384471849050706, 'comet_planet_ids': array([], dtype=int64), 'comets': array([], dtype=object), 'fleets': array([], dtype=object), 'initial_planets': array([array([ 0.        , -1.        , 96.92510434, 62.6384318 ,  2.60943791,
              11.        ,  5.        ])                                      ,
       array([ 1.        , -1.        , 37.3615682 , 96.92510434,  2.60943791,
              11.        ,  5.        ])                                      ,
       array([ 2.        , -1.        , 62.6384318 ,  3.07489566,  2.60943791,
              11.        ,  5.        ])                                      ,
       array([ 3.        , -1.        ,  3.07489566, 37.3615682 ,  2.60943791,
              11.        ,  5.        ])                                      ,
       array([ 4.        , -1.        , 78.25172464, 90.81098899,  2.09861229,
              26.        ,  3.        ])                                      ,
       array([

In [7]:
from datasets import replay_to_dataframe, _normalise_paths
import pandas as pd
df_json = replay_to_dataframe("/home/tts26/sonh/Orbit-Wars-RL/gnn_rl/replays/77249937.json")

print(f"Data Shape: {df_json.shape}")
df_json.dtypes

/home/tts26/miniconda3/envs/orbit_wars/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data Shape: (315, 7)


step               int64
obs_step           int64
player_id          int64
action_source    float64
action_angle     float64
action_ships     float64
raw_obs           object
dtype: object

In [8]:
sample = df_json["raw_obs"].iloc[0]

print(type(sample))
print(sample)

<class 'dict'>
{'angular_velocity': 0.028384471849050706, 'comet_planet_ids': [], 'comets': [], 'fleets': [], 'initial_planets': [[0, -1, 96.92510434311379, 62.63843180081058, 2.6094379124341005, 11, 5], [1, -1, 37.36156819918942, 96.92510434311379, 2.6094379124341005, 11, 5], [2, -1, 62.63843180081058, 3.0748956568862127, 2.6094379124341005, 11, 5], [3, -1, 3.0748956568862127, 37.36156819918942, 2.6094379124341005, 11, 5], [4, -1, 78.251724635019, 90.81098898515404, 2.09861228866811, 26, 3], [5, -1, 9.18901101484596, 78.251724635019, 2.09861228866811, 26, 3], [6, -1, 90.81098898515404, 21.748275364980998, 2.09861228866811, 26, 3], [7, -1, 21.748275364980998, 9.18901101484596, 2.09861228866811, 26, 3], [8, -1, 93.03771089891809, 93.00732023145017, 2.6094379124341005, 87, 5], [9, -1, 6.992679768549834, 93.03771089891809, 2.6094379124341005, 87, 5], [10, -1, 93.00732023145017, 6.9622891010819075, 2.6094379124341005, 87, 5], [11, -1, 6.9622891010819075, 6.992679768549834, 2.60943791243410

In [ ]:
%load_ext autoreload
%autoreload 2
import torch
from kaggle_environments import make
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from kaggle_agent_wrapper import KaggleGNNWrapper
from rl_model import OrbitWarsGraphBuilder
from models import GNNAgent
from ppo import Agent
from minhtuuse import AggressiveNearestAgent

env = make("orbit_wars", configuration={"seed": 2}, debug=True)

opponent = AggressiveNearestAgent()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Agent(
        node_dim=21, edge_dim=15, global_dim=21, 
        hidden_dim=256, num_layers=3, num_ship_buckets=20
    ).to(device)

# model = GNNAgent(
#     node_dim=21, edge_dim=15, global_dim=21, 
#     hidden_dim=256, num_layers=3, num_ship_buckets=20
# )
checkpoint_path = "/home/tts26/sonh/Orbit-Wars-RL/gnn_rl/models/ppo_agent_300.pth" 
print(f"Loading weights from {checkpoint_path}...")
try:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=False)

except Exception as e:
    print(f"Warning: Không thể load weights ({e}). Đang chạy bằng mô hình random để test luồng.")

trained_agent = KaggleGNNWrapper(model, device, graph_builder=OrbitWarsGraphBuilder())

print("Starting evaluation match...")
env.run([trained_agent.act, opponent.act])

final = env.steps[-1]
print("\n--- Match Results ---")
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

env.render(mode="ipython", width=800, height=600)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 23.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_s

/home/tts26/miniconda3/envs/orbit_wars/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights from /home/tts26/sonh/Orbit-Wars-RL/gnn_rl/models/ppo_agent_200.pth...
Starting evaluation match...

--- Match Results ---
Player 0: reward=-1, status=DONE
Player 1: reward=1, status=DONE


In [4]:
import pandas as pd

def analyze_parquet_dataset(parquet_path: str):
    print(f"Đang tải dữ liệu từ: {parquet_path} ...\n")
    
    # 1. Đọc file Parquet (Yêu cầu cài đặt pyarrow hoặc fastparquet)
    # pip install pandas pyarrow
    df = pd.read_parquet(parquet_path)
    
    # 2. In ra tổng quan
    print("="*50)
    print(" 1. TỔNG QUAN DỮ LIỆU")
    print("="*50)
    print(f"Tổng số samples (rows): {len(df):,}")
    print(f"Danh sách các cột: {list(df.columns)}\n")
    
    # 3. In ra 5 dòng đầu tiên để xem hình thù dữ liệu
    print("="*50)
    print(" 2. MỘT SỐ MẪU DỮ LIỆU (5 dòng đầu)")
    print("="*50)
    # Nếu có cột raw_obs dạng dict/list quá dài, bạn có thể drop tạm khi in để dễ nhìn:
    # print(df.drop(columns=['raw_obs'], errors='ignore').head())
    print(df.head(), "\n")
    
    # 4. Thống kê phân bố của ship_pct và angle
    target_cols = ['ship_pct', 'angle']
    # Lọc ra các cột thực sự tồn tại trong dataframe (đề phòng bạn đặt tên khác)
    existing_cols = [col for col in target_cols if col in df.columns]
    
    if existing_cols:
        print("="*50)
        print(" 3. THỐNG KÊ MÔ TẢ (Min, Max, Mean, Std)")
        print("="*50)
        print(df[existing_cols].describe(), "\n")
        
        print("="*50)
        print(" 4. PHÂN BỐ DỮ LIỆU (HISTOGRAM BINS)")
        print("="*50)
        for col in existing_cols:
            print(f"\n--- Phân bố của cột '{col}' ---")
            
            # Chia dữ liệu thành 10 khoảng (bins) bằng nhau giống histogram
            binned_counts = pd.cut(df[col], bins=10).value_counts().sort_index()
            
            # Tính phần trăm
            total_samples = len(df)
            
            # In ra dạng bảng text dễ nhìn
            print(f"{'Khoảng giá trị (Bin)':<25} | {'Số lượng':<10} | {'Tỷ lệ %':<10}")
            print("-" * 50)
            for interval, count in binned_counts.items():
                pct = (count / total_samples) * 100
                print(f"{str(interval):<25} | {count:<10,} | {pct:.2f}%")
    else:
        print(f"⚠️ Không tìm thấy các cột {target_cols} để thống kê.")

# --- Cách sử dụng ---
if __name__ == "__main__":
    # Thay đường dẫn tới file parquet của bạn vào đây
    analyze_parquet_dataset("/home/tts26/sonh/Orbit-Wars-RL/gnn_rl/player_flg_chunks/chunk_000.parquet")

Đang tải dữ liệu từ: /home/tts26/sonh/Orbit-Wars-RL/gnn_rl/player_flg_chunks/chunk_000.parquet ...

 1. TỔNG QUAN DỮ LIỆU
Tổng số samples (rows): 1,855,301
Danh sách các cột: ['step', 'obs_step', 'player_id', 'action_source', 'action_angle', 'action_ships', 'ship_deductions', 'simulated_fleets', 'raw_obs']

 2. MỘT SỐ MẪU DỮ LIỆU (5 dòng đầu)
   step  obs_step  player_id  action_source  action_angle  action_ships  \
0     2         1          0              8     -2.059017          15.0   
1     2         1          0             -1      0.000000           0.0   
2     6         5          0              8      2.195751          20.0   
3     6         5          0             -1      0.000000           0.0   
4    10         9          0              8     -2.121447          20.0   

                                     ship_deductions  \
0  {'8': None, '18': None, '12': None, '0': None,...   
1  {'8': 15.0, '18': None, '12': None, '0': None,...   
2  {'8': None, '18': None, '12': Non